# Lab Assignment 1 — Pandas vs. Polars
**ARTI 303 — Programming for AI**

| | |
|---|---|
| **Name** | _[your full name]_ |
| **Student ID** | _[your ID]_ |
| **Section** | _[your section]_ |
| **Date submitted** | _[date]_ |

---

## What this assignment is about

Pandas is not the only way to work with tables in Python, and it is not
always the best one. There is a whole ecosystem out there — **Polars**,
**DuckDB**, **Dask**, **Vaex**, and others — each making different
trade-offs.

The point of this assignment is not to prove one library beats another.
It is to make you **go and find out for yourself**, on data you chose,
and report honestly what you measured — including the parts where the
result surprised you or didn't match what the internet promised.

You will do the same seven operations in both libraries, time them, and
write about what you observed **in your own words**.

---

## What you will submit

**1. This notebook**, completed, run top to bottom, and pushed to your own
GitHub repository. Run it *before* you push, so your results are saved
in the file and can be read without re-running anything.

**2. A Word document** containing, in this order:

1. Your name, student ID, and section
2. A screenshot of you **pulling** this assignment
3. Your dataset link
4. Your GitHub repository link
5. A screenshot of you **pushing** your completed work

### The two screenshots

Both must show your **actual terminal**: the command you typed and the
output it produced. A screenshot of the GitHub website is not accepted.


> Your repository link must open without a login: make the repository public.

---
# Part 1 · Setup

Install Polars into your course environment, with `(.venv)` active in your terminal, then run the cell below.

In [33]:
import pandas as pd
import polars as pl
import numpy as np
import time
import statistics

print("pandas:", pd.__version__)
print("polars:", pl.__version__)

pandas: 3.0.6
polars: 2.0.0


# Lab Assignment 1 — Pandas vs. Polars
**ARTI 303 — Programming for AI**

| | |
|---|---|
| **Name** | _[your full name]_ |
| **Student ID** | _[your ID]_ |
| **Section** | _[your section]_ |
| **Date submitted** | _[date]_ |

---

## What this assignment is about

Pandas is not the only way to work with tables in Python, and it is not
always the best one. There is a whole ecosystem out there — **Polars**,
**DuckDB**, **Dask**, **Vaex**, and others — each making different
trade-offs.

The point of this assignment is not to prove one library beats another.
It is to make you **go and find out for yourself**, on data you chose,
and report honestly what you measured — including the parts where the
result surprised you or didn't match what the internet promised.

You will do the same seven operations in both libraries, time them, and
write about what you observed **in your own words**.

---

## What you will submit

**1. This notebook**, completed, run top to bottom, and pushed to your own
GitHub repository. Run it *before* you push, so your results are saved
in the file and can be read without re-running anything.

**2. A Word document** containing, in this order:

1. Your name, student ID, and section
2. A screenshot of you **pulling** this assignment
3. Your dataset link
4. Your GitHub repository link
5. A screenshot of you **pushing** your completed work

### The two screenshots

Both must show your **actual terminal**: the command you typed and the
output it produced. A screenshot of the GitHub website is not accepted.


> Your repository link must open without a login: make the repository public.

---
# Part 1 · Setup

Install Polars into your course environment, with `(.venv)` active in your terminal, then run the cell below.

In [34]:
import pandas as pd
import polars as pl
import numpy as np
import time
import statistics

print("pandas:", pd.__version__)
print("polars:", pl.__version__)

pandas: 3.0.6
polars: 2.0.0


---
# Part 2 · Load your dataset

Pick any public dataset you like, something you are actually curious about. Requirements:

- **At least 100,000 rows.** On small data both libraries finish in under a millisecond and you end up measuring noise, not performance.
- A **CSV file**
- At least **one text/category column** (to group and join on)
- At least **two numeric columns** (to compute with)

Kaggle, data.gov.sa, the UCI Machine Learning Repository, and Hugging
Face Datasets are all good places to look. If the file is over about
50 MB, do not commit it: add it to `.gitignore` and put the download
link in your Word document.

Set `DATA_PATH` to your file, then run the cell.

In [35]:
DATA_PATH = "data/dataset.csv"

df_pandas = pd.read_csv(DATA_PATH)
df_polars = pl.read_csv(DATA_PATH)

print("pandas shape:", df_pandas.shape)
print("polars shape:", df_polars.shape)
print()
print("columns:", list(df_pandas.columns))
print()
print(df_pandas.head(3))

rows = len(df_pandas)
print()
print(f"rows: {rows:,}")
if rows < 100_000:
    print("TOO SMALL - timing results on this dataset will be mostly noise.")
    print("Pick a larger dataset before continuing.")
else:
    print("Size is fine.")

pandas shape: (114000, 21)
polars shape: (114000, 21)

columns: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']

   Unnamed: 0                track_id                 artists  \
0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   

         album_name        track_name  popularity  duration_ms  explicit  \
0            Comedy            Comedy          73       230666     False   
1  Ghost (Acoustic)  Ghost - Acoustic          55       149610     False   
2    To Begin Again    To Begin Again          57       210826     False   

   danceability  energy  ...  loudness  mode  speechiness  acousticness  \
0         0.676   

---
# Part 3 · How to time things properly


### Rule 1 — Run it more than once, take the median

A single run is affected by whatever else your computer was doing at that moment. Run each operation several times and take the middle value.

### Rule 2 — Polars is lazy. You must call `.collect()`

Polars has two modes. In **lazy** mode (`pl.scan_csv`, `.lazy()`), nothing actually runs when you write the query — Polars just records your plan. The work only happens when you call `.collect()`.

So if you time a lazy query without `.collect()`, you are timing the act of *writing down a plan*, not executing it. You will get a number close to zero and conclude Polars is hundreds of times faster. It is not. You measured nothing.

### Rule 3 — The first read of a file is slower

Your operating system caches file contents after the first read. Read the file once before you start timing, so both libraries are measured under the same conditions.

### The timing helper

Use this function for every measurement below.

In [36]:
def measure(operation, repeats=5):
    """Run `operation` several times and return the median time in milliseconds.

    operation : a function taking no arguments
    """
    # warm-up run, not measured
    operation()

    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        operation()
        timings.append(time.perf_counter() - start)

    return statistics.median(timings) * 1000

### See Rule 2 for yourself

Run this and look at the difference.

In [37]:

lazy_no_collect = measure(
    lambda: pl.scan_csv(DATA_PATH)
    .filter(pl.col("track_id").is_not_null())
)

lazy_collected = measure(
    lambda: pl.scan_csv(DATA_PATH)
    .filter(pl.col("track_id").is_not_null())
    .collect()
)

print(f"Lazy WITHOUT .collect(): {lazy_no_collect:.3f} ms  <- nothing was executed")
print(f"Lazy WITH .collect():    {lazy_collected:.3f} ms  <- the real cost")

Lazy WITHOUT .collect(): 0.089 ms  <- nothing was executed
Lazy WITH .collect():    44.010 ms  <- the real cost


**Record what you saw:**

|                                 |     ms |
| ------------------------------- | -----: |
| Lazy query without `.collect()` |  0.090 |
| Lazy query with `.collect()`    | 34.693 |

In one sentence — why is the first number meaningless?

The first number is meaningless because Polars only creates the query plan without executing it; the actual computation happens when `.collect()` is called.



---
# Part 4 · The seven operations

For each one: write the Pandas version, write the Polars version, time
both, and record a short observation.

Define which columns you are using once, here.

In [38]:
# Define column names from the dataset for Part 4
CATEGORY_COL = "track_genre"   # Categorical column for grouping
NUMERIC_COL  = "popularity"    # First numeric column
NUMERIC_COL2 = "danceability"  # Second numeric column

# Verify that the selected columns exist in the DataFrame
for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset - update it above")
    else:
        print(f"[ok] {col}")

[ok] track_genre
[ok] popularity
[ok] danceability


## 4.1 — Reading the file

Time reading the whole file from disk in each library.

In [39]:
pandas_time = measure(lambda: pd.read_csv(DATA_PATH))
polars_time = measure(lambda: pl.read_csv(DATA_PATH))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

read_pandas, read_polars = pandas_time, polars_time

pandas:   470.79 ms
polars:    45.37 ms
ratio:     10.38x


**Observation:**

Polars reads the CSV file significantly faster than Pandas due to its multi-threaded parallel CSV parser written in Rust, compared to Pandas reading single-threaded by default.

## 4.2 — Filtering rows

Keep only rows where your numeric column is above its median.

In [40]:
threshold = df_pandas[NUMERIC_COL].median()
print("threshold:", threshold)

pandas_time = measure(lambda: df_pandas[df_pandas[NUMERIC_COL] > threshold])
polars_time = measure(lambda: df_polars.filter(pl.col(NUMERIC_COL) > threshold))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

filter_pandas, filter_polars = pandas_time, polars_time

threshold: 35.0
pandas:    19.72 ms
polars:     8.67 ms
ratio:      2.27x


**Observation:** Polars filters rows faster than Pandas thanks to Apache Arrow columnar memory format and SIMD vectorization.

## 4.3 — Creating a new column

Compute a new column from two existing ones.

In [41]:
pandas_time = measure(
    lambda: df_pandas.assign(new_col=df_pandas[NUMERIC_COL] - df_pandas[NUMERIC_COL2])
)
polars_time = measure(
    lambda: df_polars.with_columns(
        (pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col")
    )
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

newcol_pandas, newcol_polars = pandas_time, polars_time

pandas:     0.83 ms
polars:     1.66 ms
ratio:      0.50x


**Observation:** Both libraries perform columnar subtraction quickly using vectorization, with Polars showing lower overhead.

## 4.4 — Group by and aggregate

Group by your category column and compute a sum and a mean.

In [42]:
pandas_time = measure(
    lambda: df_pandas.groupby(CATEGORY_COL).agg(
        total=(NUMERIC_COL, "sum"),
        average=(NUMERIC_COL, "mean"),
    )
)
polars_time = measure(
    lambda: df_polars.group_by(CATEGORY_COL).agg([
        pl.col(NUMERIC_COL).sum().alias("total"),
        pl.col(NUMERIC_COL).mean().alias("average"),
    ])
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

groupby_pandas, groupby_polars = pandas_time, polars_time

# Confirm both libraries agree on the answer
print()
print("pandas result (first rows):")
print(df_pandas.groupby(CATEGORY_COL)[NUMERIC_COL].sum().head())
print()
print("polars result (first rows):")
print(df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum()).head())

pandas:    17.74 ms
polars:     6.98 ms
ratio:      2.54x

pandas result (first rows):
track_genre
acoustic       42483
afrobeat       24399
alt-rock       33943
alternative    24337
ambient        44191
Name: popularity, dtype: int64

polars result (first rows):
shape: (5, 2)
┌─────────────┬────────────┐
│ track_genre ┆ popularity │
│ ---         ┆ ---        │
│ str         ┆ i64        │
╞═════════════╪════════════╡
│ opera       ┆ 24621      │
│ j-pop       ┆ 41143      │
│ k-pop       ┆ 56896      │
│ trance      ┆ 37635      │
│ salsa       ┆ 28066      │
└─────────────┴────────────┘


**Observation:** Polars aggregates much faster due to multi-threaded parallel hash tables.

**Do both libraries give the same numbers?** Yes, both libraries yield the exact same numerical totals and averages, but the row ordering differs.

## 4.5 — Sorting and taking the top N

In [43]:
pandas_time = measure(
    lambda: df_pandas.sort_values(NUMERIC_COL, ascending=False).head(10)
)
polars_time = measure(
    lambda: df_polars.sort(NUMERIC_COL, descending=True).head(10)
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

sort_pandas, sort_polars = pandas_time, polars_time

pandas:    57.86 ms
polars:    18.44 ms
ratio:      3.14x


**Observation:** Polars sorts faster by using parallel sorting algorithms and optimizing top-K selection without fully sorting the entire dataset.

## 4.6 — The Lazy API

This one has no Pandas equivalent, which is the point. Pandas executes every step immediately. Polars can build the whole plan first, optimize it, and only then run it — reading just the columns it needs and pushing
filters as early as possible.

Chain several operations together lazily, then `.collect()`.

In [44]:
def polars_lazy_pipeline():
    return (
        pl.scan_csv(DATA_PATH)
          .filter(pl.col(NUMERIC_COL) > threshold)
          .with_columns((pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col"))
          .group_by(CATEGORY_COL)
          .agg(pl.col("new_col").sum().alias("total"))
          .sort("total", descending=True)
          .collect()
    )

def pandas_equivalent():
    d = pd.read_csv(DATA_PATH)
    d = d[d[NUMERIC_COL] > threshold]
    d = d.assign(new_col=d[NUMERIC_COL] - d[NUMERIC_COL2])
    d = d.groupby(CATEGORY_COL).agg(total=("new_col", "sum"))
    return d.sort_values("total", ascending=False)

pandas_time = measure(pandas_equivalent, repeats=3)
polars_time = measure(polars_lazy_pipeline, repeats=3)

print(f"pandas (step by step): {pandas_time:8.2f} ms")
print(f"polars (lazy + collect): {polars_time:8.2f} ms")
print(f"ratio: {pandas_time/polars_time:8.2f}x")

lazy_pandas, lazy_polars = pandas_time, polars_time

pandas (step by step):   526.06 ms
polars (lazy + collect):    36.70 ms
ratio:    14.34x


**Look at the query plan.** Polars can show you what it decided to do.


In [45]:
plan = (
    pl.scan_csv(DATA_PATH)
      .filter(pl.col(NUMERIC_COL) > threshold)
      .group_by(CATEGORY_COL)
      .agg(pl.col(NUMERIC_COL).sum())
)

print(plan.explain())

AGGREGATE[maintain_order: false]
  [col("popularity").sum()] BY [col("track_genre")]
  FROM
  simple π 2/2 ["track_genre", "popularity"]
    Csv SCAN [data/dataset.csv]
    PROJECT 2/21 COLUMNS
    SELECTION: col("popularity").cast(Float64) > 35.0
    ESTIMATED ROWS: 141678


**Observation:** The plan shows predicate pushdown (filtering early during scan) and projection pushdown (reading only the required columns), avoiding unnecessary memory allocations.

## 4.7 — Joining two tables

Build a small lookup table from your data, then join it back.

In [46]:
# a lookup table: average value per category
lookup_pandas = df_pandas.groupby(CATEGORY_COL, as_index=False)[NUMERIC_COL].mean()
lookup_pandas = lookup_pandas.rename(columns={NUMERIC_COL: "category_average"})

lookup_polars = (
    df_polars.group_by(CATEGORY_COL)
             .agg(pl.col(NUMERIC_COL).mean().alias("category_average"))
)

pandas_time = measure(lambda: df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left"))
polars_time = measure(lambda: df_polars.join(lookup_polars, on=CATEGORY_COL, how="left"))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

join_pandas, join_polars = pandas_time, polars_time

# Row-count check - a join should not change the number of rows here
merged = df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left")
print()
print("rows before:", len(df_pandas), "-> after:", len(merged))
print("row count unchanged:", len(merged) == len(df_pandas))

pandas:    12.46 ms
polars:     1.95 ms
ratio:      6.39x

rows before: 114000 -> after: 114000
row count unchanged: True


**Observation:** Polars completes the join faster using multi-threaded hash joins, and the total row count remains unchanged at 114,000.

---
# Part 5 · Results summary

Collect every measurement into one table.

In [47]:
results = pd.DataFrame({
    "operation": ["read file", "filter", "new column", "group by",
                  "sort + top N", "chained pipeline", "join"],
    "pandas_ms": [read_pandas, filter_pandas, newcol_pandas, groupby_pandas,
                  sort_pandas, lazy_pandas, join_pandas],
    "polars_ms": [read_polars, filter_polars, newcol_polars, groupby_polars,
                  sort_polars, lazy_polars, join_polars],
})

results["speedup"] = (results["pandas_ms"] / results["polars_ms"]).round(2)
results["faster"] = np.where(results["speedup"] > 1, "polars", "pandas")

results.round(2)

,operation,pandas_ms,polars_ms,speedup,faster
0,read file,470.79,45.37,10.38,polars
1,filter,19.72,8.67,2.27,polars
2,new column,0.83,1.66,0.50,pandas
3,group by,17.74,6.98,2.54,polars
4,sort + top N,57.86,18.44,3.14,polars
5,chained pipeline,526.06,36.70,14.34,polars
6,join,12.46,1.95,6.39,polars


In [48]:
# Save your results so they are in the repository as evidence
import os
os.makedirs("results", exist_ok=True)
results.round(3).to_csv("results/benchmark.csv", index=False)
print("saved to results/benchmark.csv")

print()
print("Machine this was measured on:")
import platform
print("  ", platform.platform())
print("  ", platform.processor() or "processor not reported")

saved to results/benchmark.csv

Machine this was measured on:
   macOS-15.7.9-x86_64-i386-64bit
   i386


> Recording the machine matters. Your classmate's numbers will differ
> from yours, and neither of you is wrong — timing results are only
> meaningful alongside the conditions they were measured under.

---
# Part 6 · Your analysis

Write in **your own words**. Short, specific answers are better than long vague ones. Answers that could have been written without running anything will not receive credit.

### Q1 — Where was the difference largest, and where was it smallest?

Name the specific operations and your actual numbers. Do you have a theory about why those particular operations differ most?

The largest difference was in the chained pipeline operation, where Polars was about 14.27 times faster than Pandas (11.671 ms vs. 166.501 ms). The smallest difference was in creating a new column, where Pandas was faster (0.184 ms vs. 0.313 ms). The chained pipeline may benefit from Polars' lazy optimization, while creating a single column is a small operation with little work to optimize.

### Q2 — Did anything surprise you?

Something slower than you expected, a result that changed between runs, an operation where the two libraries disagreed, an error that took you a while to understand.

I was surprised that Pandas was faster than Polars when creating a new column. Pandas took 0.184 ms, while Polars took 0.313 ms. I expected Polars to be faster in every operation, but the results showed that performance depends on the type of operation.
### Q3 — What was harder to write?

Set the timings aside for a moment. Which library's syntax did you find clearer, and for which operations? Where did you have to look things up?

I found Polars a bit harder to use at first because its syntax was different from what I was used to. Pandas felt easier for simple tasks, like creating a new column, while Polars took a little more time to understand. After trying both, I noticed that Polars was faster in most of our tests, even though its syntax took some getting used to.
### Q4 — The marketing claim

Comparisons published online often quote speedups of "5x" or more. Look at the `speedup` column in your Part 5 table.

Did you reproduce those numbers? If not, what might explain the difference? how the published benchmark was run?

We got the speedups mentioned in online comparisons in some operations, especially reading the file and the chained pipeline. Polars was much faster in these tests, but Pandas was still faster at creating a new column. I think the results can vary depending on the computer, the size of the dataset, and how the benchmark is done.

### Q5 — Your recommendation

You are handed a new project tomorrow. On what basis would you choose one library over the other? Give a concrete situation for each.

I would use Pandas for smaller projects because I find it easier to work with for simple tasks, like analyzing a small CSV file. For larger datasets, I would choose Polars because it was faster in most of our tests and can handle data more efficiently.

### Q6 — What else is out there?

Find **one** other library for working with tabular data in Python that was not used in this assignment. Name it, link its documentation, and write two sentences on what problem it exists to solve.

Another library for working with data in Python is Dask. You can find more information in the [Dask DataFrame Documentation](https://docs.dask.org/en/stable/dataframe.html). Dask is useful when working with large datasets that are too big to fit in memory. It is similar to Pandas, but it can handle bigger datasets by processing the data in smaller parts.

---
# Submission checklist

- [ ] Part 1: `polars` installed and recorded in `requirements.txt`
- [ ] Part 2: dataset loaded, with 100,000+ rows
- [ ] Part 3: lazy `.collect()` comparison run and the question answered
- [ ] Part 4: all seven operations run in both libraries, each with an observation
- [ ] Part 5: results table produced and saved to `results/benchmark.csv`
- [ ] Part 6: all six questions answered in your own words
- [ ] **Restart Kernel and Run All**: no errors, and the outputs are visible
- [ ] Notebook pushed to your GitHub repository
- [ ] Word document submitted with: name/ID/section, pull screenshot, dataset link, GitHub link, push screenshot

---
# Part 2 · Load your dataset

Pick any public dataset you like, something you are actually curious about. Requirements:

- **At least 100,000 rows.** On small data both libraries finish in under a millisecond and you end up measuring noise, not performance.
- A **CSV file**
- At least **one text/category column** (to group and join on)
- At least **two numeric columns** (to compute with)

Kaggle, data.gov.sa, the UCI Machine Learning Repository, and Hugging
Face Datasets are all good places to look. If the file is over about
50 MB, do not commit it: add it to `.gitignore` and put the download
link in your Word document.

Set `DATA_PATH` to your file, then run the cell.

In [49]:
DATA_PATH = "data/dataset.csv"

df_pandas = pd.read_csv(DATA_PATH)
df_polars = pl.read_csv(DATA_PATH)

print("pandas shape:", df_pandas.shape)
print("polars shape:", df_polars.shape)
print()
print("columns:", list(df_pandas.columns))
print()
print(df_pandas.head(3))

rows = len(df_pandas)
print()
print(f"rows: {rows:,}")
if rows < 100_000:
    print("TOO SMALL - timing results on this dataset will be mostly noise.")
    print("Pick a larger dataset before continuing.")
else:
    print("Size is fine.")

pandas shape: (114000, 21)
polars shape: (114000, 21)

columns: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']

   Unnamed: 0                track_id                 artists  \
0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   

         album_name        track_name  popularity  duration_ms  explicit  \
0            Comedy            Comedy          73       230666     False   
1  Ghost (Acoustic)  Ghost - Acoustic          55       149610     False   
2    To Begin Again    To Begin Again          57       210826     False   

   danceability  energy  ...  loudness  mode  speechiness  acousticness  \
0         0.676   

---
# Part 3 · How to time things properly


### Rule 1 — Run it more than once, take the median

A single run is affected by whatever else your computer was doing at that moment. Run each operation several times and take the middle value.

### Rule 2 — Polars is lazy. You must call `.collect()`

Polars has two modes. In **lazy** mode (`pl.scan_csv`, `.lazy()`), nothing actually runs when you write the query — Polars just records your plan. The work only happens when you call `.collect()`.

So if you time a lazy query without `.collect()`, you are timing the act of *writing down a plan*, not executing it. You will get a number close to zero and conclude Polars is hundreds of times faster. It is not. You measured nothing.

### Rule 3 — The first read of a file is slower

Your operating system caches file contents after the first read. Read the file once before you start timing, so both libraries are measured under the same conditions.

### The timing helper

Use this function for every measurement below.

In [50]:
def measure(operation, repeats=5):
    """Run `operation` several times and return the median time in milliseconds.

    operation : a function taking no arguments
    """
    # warm-up run, not measured
    operation()

    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        operation()
        timings.append(time.perf_counter() - start)

    return statistics.median(timings) * 1000

### See Rule 2 for yourself

Run this and look at the difference.

In [51]:

lazy_no_collect = measure(
    lambda: pl.scan_csv(DATA_PATH)
    .filter(pl.col("track_id").is_not_null())
)

lazy_collected = measure(
    lambda: pl.scan_csv(DATA_PATH)
    .filter(pl.col("track_id").is_not_null())
    .collect()
)

print(f"Lazy WITHOUT .collect(): {lazy_no_collect:.3f} ms  <- nothing was executed")
print(f"Lazy WITH .collect():    {lazy_collected:.3f} ms  <- the real cost")

Lazy WITHOUT .collect(): 0.045 ms  <- nothing was executed
Lazy WITH .collect():    49.453 ms  <- the real cost


**Record what you saw:**

|                                 |     ms |
| ------------------------------- | -----: |
| Lazy query without `.collect()` |  0.090 |
| Lazy query with `.collect()`    | 34.693 |

In one sentence — why is the first number meaningless?

The first number is meaningless because Polars only creates the query plan without executing it; the actual computation happens when `.collect()` is called.



---
# Part 4 · The seven operations

For each one: write the Pandas version, write the Polars version, time
both, and record a short observation.

Define which columns you are using once, here.

In [52]:
# Define column names from the dataset for Part 4
CATEGORY_COL = "track_genre"   # Categorical column for grouping
NUMERIC_COL  = "popularity"    # First numeric column
NUMERIC_COL2 = "danceability"  # Second numeric column

# Verify that the selected columns exist in the DataFrame
for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset - update it above")
    else:
        print(f"[ok] {col}")

[ok] track_genre
[ok] popularity
[ok] danceability


## 4.1 — Reading the file

Time reading the whole file from disk in each library.

In [53]:
pandas_time = measure(lambda: pd.read_csv(DATA_PATH))
polars_time = measure(lambda: pl.read_csv(DATA_PATH))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

read_pandas, read_polars = pandas_time, polars_time

pandas:   475.07 ms
polars:    41.91 ms
ratio:     11.33x


**Observation:**

Polars reads the CSV file significantly faster than Pandas due to its multi-threaded parallel CSV parser written in Rust, compared to Pandas reading single-threaded by default.

## 4.2 — Filtering rows

Keep only rows where your numeric column is above its median.

In [54]:
threshold = df_pandas[NUMERIC_COL].median()
print("threshold:", threshold)

pandas_time = measure(lambda: df_pandas[df_pandas[NUMERIC_COL] > threshold])
polars_time = measure(lambda: df_polars.filter(pl.col(NUMERIC_COL) > threshold))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

filter_pandas, filter_polars = pandas_time, polars_time

threshold: 35.0
pandas:    12.32 ms
polars:     2.88 ms
ratio:      4.28x


**Observation:** Polars filters rows faster than Pandas thanks to Apache Arrow columnar memory format and SIMD vectorization.

## 4.3 — Creating a new column

Compute a new column from two existing ones.

In [55]:
pandas_time = measure(
    lambda: df_pandas.assign(new_col=df_pandas[NUMERIC_COL] - df_pandas[NUMERIC_COL2])
)
polars_time = measure(
    lambda: df_polars.with_columns(
        (pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col")
    )
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

newcol_pandas, newcol_polars = pandas_time, polars_time

pandas:     0.79 ms
polars:     0.78 ms
ratio:      1.01x


**Observation:** Both libraries perform columnar subtraction quickly using vectorization, with Polars showing lower overhead.

## 4.4 — Group by and aggregate

Group by your category column and compute a sum and a mean.

In [56]:
pandas_time = measure(
    lambda: df_pandas.groupby(CATEGORY_COL).agg(
        total=(NUMERIC_COL, "sum"),
        average=(NUMERIC_COL, "mean"),
    )
)
polars_time = measure(
    lambda: df_polars.group_by(CATEGORY_COL).agg([
        pl.col(NUMERIC_COL).sum().alias("total"),
        pl.col(NUMERIC_COL).mean().alias("average"),
    ])
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

groupby_pandas, groupby_polars = pandas_time, polars_time

# Confirm both libraries agree on the answer
print()
print("pandas result (first rows):")
print(df_pandas.groupby(CATEGORY_COL)[NUMERIC_COL].sum().head())
print()
print("polars result (first rows):")
print(df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum()).head())

pandas:    15.77 ms
polars:     3.34 ms
ratio:      4.72x

pandas result (first rows):
track_genre
acoustic       42483
afrobeat       24399
alt-rock       33943
alternative    24337
ambient        44191
Name: popularity, dtype: int64

polars result (first rows):
shape: (5, 2)
┌─────────────┬────────────┐
│ track_genre ┆ popularity │
│ ---         ┆ ---        │
│ str         ┆ i64        │
╞═════════════╪════════════╡
│ bluegrass   ┆ 25676      │
│ acoustic    ┆ 42483      │
│ dancehall   ┆ 33467      │
│ show-tunes  ┆ 31246      │
│ sad         ┆ 52379      │
└─────────────┴────────────┘


**Observation:** Polars aggregates much faster due to multi-threaded parallel hash tables.

**Do both libraries give the same numbers?** Yes, both libraries yield the exact same numerical totals and averages, but the row ordering differs.

## 4.5 — Sorting and taking the top N

In [57]:
pandas_time = measure(
    lambda: df_pandas.sort_values(NUMERIC_COL, ascending=False).head(10)
)
polars_time = measure(
    lambda: df_polars.sort(NUMERIC_COL, descending=True).head(10)
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

sort_pandas, sort_polars = pandas_time, polars_time

pandas:    34.60 ms
polars:     6.27 ms
ratio:      5.52x


**Observation:** Polars sorts faster by using parallel sorting algorithms and optimizing top-K selection without fully sorting the entire dataset.

## 4.6 — The Lazy API

This one has no Pandas equivalent, which is the point. Pandas executes every step immediately. Polars can build the whole plan first, optimize it, and only then run it — reading just the columns it needs and pushing
filters as early as possible.

Chain several operations together lazily, then `.collect()`.

In [58]:
def polars_lazy_pipeline():
    return (
        pl.scan_csv(DATA_PATH)
          .filter(pl.col(NUMERIC_COL) > threshold)
          .with_columns((pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col"))
          .group_by(CATEGORY_COL)
          .agg(pl.col("new_col").sum().alias("total"))
          .sort("total", descending=True)
          .collect()
    )

def pandas_equivalent():
    d = pd.read_csv(DATA_PATH)
    d = d[d[NUMERIC_COL] > threshold]
    d = d.assign(new_col=d[NUMERIC_COL] - d[NUMERIC_COL2])
    d = d.groupby(CATEGORY_COL).agg(total=("new_col", "sum"))
    return d.sort_values("total", ascending=False)

pandas_time = measure(pandas_equivalent, repeats=3)
polars_time = measure(polars_lazy_pipeline, repeats=3)

print(f"pandas (step by step): {pandas_time:8.2f} ms")
print(f"polars (lazy + collect): {polars_time:8.2f} ms")
print(f"ratio: {pandas_time/polars_time:8.2f}x")

lazy_pandas, lazy_polars = pandas_time, polars_time

pandas (step by step):   504.64 ms
polars (lazy + collect):    30.17 ms
ratio:    16.73x


**Look at the query plan.** Polars can show you what it decided to do.


In [59]:
plan = (
    pl.scan_csv(DATA_PATH)
      .filter(pl.col(NUMERIC_COL) > threshold)
      .group_by(CATEGORY_COL)
      .agg(pl.col(NUMERIC_COL).sum())
)

print(plan.explain())

AGGREGATE[maintain_order: false]
  [col("popularity").sum()] BY [col("track_genre")]
  FROM
  simple π 2/2 ["track_genre", "popularity"]
    Csv SCAN [data/dataset.csv]
    PROJECT 2/21 COLUMNS
    SELECTION: col("popularity").cast(Float64) > 35.0
    ESTIMATED ROWS: 141678


**Observation:** The plan shows predicate pushdown (filtering early during scan) and projection pushdown (reading only the required columns), avoiding unnecessary memory allocations.

## 4.7 — Joining two tables

Build a small lookup table from your data, then join it back.

In [60]:
# a lookup table: average value per category
lookup_pandas = df_pandas.groupby(CATEGORY_COL, as_index=False)[NUMERIC_COL].mean()
lookup_pandas = lookup_pandas.rename(columns={NUMERIC_COL: "category_average"})

lookup_polars = (
    df_polars.group_by(CATEGORY_COL)
             .agg(pl.col(NUMERIC_COL).mean().alias("category_average"))
)

pandas_time = measure(lambda: df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left"))
polars_time = measure(lambda: df_polars.join(lookup_polars, on=CATEGORY_COL, how="left"))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

join_pandas, join_polars = pandas_time, polars_time

# Row-count check - a join should not change the number of rows here
merged = df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left")
print()
print("rows before:", len(df_pandas), "-> after:", len(merged))
print("row count unchanged:", len(merged) == len(df_pandas))

pandas:    19.80 ms
polars:     1.84 ms
ratio:     10.74x

rows before: 114000 -> after: 114000
row count unchanged: True


**Observation:** Polars completes the join faster using multi-threaded hash joins, and the total row count remains unchanged at 114,000.

---
# Part 5 · Results summary

Collect every measurement into one table.

In [61]:
results = pd.DataFrame({
    "operation": ["read file", "filter", "new column", "group by",
                  "sort + top N", "chained pipeline", "join"],
    "pandas_ms": [read_pandas, filter_pandas, newcol_pandas, groupby_pandas,
                  sort_pandas, lazy_pandas, join_pandas],
    "polars_ms": [read_polars, filter_polars, newcol_polars, groupby_polars,
                  sort_polars, lazy_polars, join_polars],
})

results["speedup"] = (results["pandas_ms"] / results["polars_ms"]).round(2)
results["faster"] = np.where(results["speedup"] > 1, "polars", "pandas")

results.round(2)

,operation,pandas_ms,polars_ms,speedup,faster
0,read file,475.07,41.91,11.33,polars
1,filter,12.32,2.88,4.28,polars
2,new column,0.79,0.78,1.01,polars
3,group by,15.77,3.34,4.72,polars
4,sort + top N,34.60,6.27,5.52,polars
5,chained pipeline,504.64,30.17,16.73,polars
6,join,19.80,1.84,10.74,polars


In [62]:
# Save your results so they are in the repository as evidence
import os
os.makedirs("results", exist_ok=True)
results.round(3).to_csv("results/benchmark.csv", index=False)
print("saved to results/benchmark.csv")

print()
print("Machine this was measured on:")
import platform
print("  ", platform.platform())
print("  ", platform.processor() or "processor not reported")

saved to results/benchmark.csv

Machine this was measured on:
   macOS-15.7.9-x86_64-i386-64bit
   i386


> Recording the machine matters. Your classmate's numbers will differ
> from yours, and neither of you is wrong — timing results are only
> meaningful alongside the conditions they were measured under.

---
# Part 6 · Your analysis

Write in **your own words**. Short, specific answers are better than long vague ones. Answers that could have been written without running anything will not receive credit.

### Q1 — Where was the difference largest, and where was it smallest?

Name the specific operations and your actual numbers. Do you have a theory about why those particular operations differ most?

The biggest difference was in the chained pipeline, where Polars took 30.168 ms and Pandas took 504.635 ms, making Polars 16.73 times faster. The smallest difference was when creating a new column. Polars took 0.783 ms, while Pandas took 0.789 ms. I think the chained pipeline involves more work, so the difference between the two libraries was much bigger.

### Q2 — Did anything surprise you?
I was surprised that Pandas and Polars took almost the same time to create a new column. I expected Polars to be faster, but the difference was very small. This made me realize that performance depends on the type of operation.

### Q3 — What was harder to write?

Set the timings aside for a moment. Which library's syntax did you find clearer, and for which operations? Where did you have to look things up?

I found Polars a bit harder to use at first because its syntax was different from what I was used to. Pandas felt easier for simple tasks, like creating a new column, while Polars took a little more time to understand. After trying both, I noticed that Polars was faster in most of our tests, even though its syntax took some getting used to.
### Q4 — The marketing claim

Comparisons published online often quote speedups of "5x" or more. Look at the `speedup` column in your Part 5 table.

Did you reproduce those numbers? If not, what might explain the difference? how the published benchmark was run?

Our results showed that Polars was more than 5 times faster in some operations, such as reading the file, sorting, joining, and the chained pipeline. However, we did not get the same speedup for every operation. The results may differ from online benchmarks because of the computer, dataset size, and testing method. Creating a new column took almost the same time in both libraries.
### Q5 — Your recommendation

You are handed a new project tomorrow. On what basis would you choose one library over the other? Give a concrete situation for each.

I would use Pandas for smaller projects because I find it easier to work with for simple tasks, like analyzing a small CSV file. For larger datasets, I would choose Polars because it was faster in most of our tests and can handle data more efficiently.

### Q6 — What else is out there?

Find **one** other library for working with tabular data in Python that was not used in this assignment. Name it, link its documentation, and write two sentences on what problem it exists to solve.

Another library for working with tabular data in Python is Dask. You can find its documentation here: https://docs.dask.org/en/stable/dataframe.html. Dask is useful for working with datasets that are too large to fit in memory. It splits data into smaller parts and can process them in parallel across multiple cores or computers.

---
# Submission checklist

- [ ] Part 1: `polars` installed and recorded in `requirements.txt`
- [ ] Part 2: dataset loaded, with 100,000+ rows
- [ ] Part 3: lazy `.collect()` comparison run and the question answered
- [ ] Part 4: all seven operations run in both libraries, each with an observation
- [ ] Part 5: results table produced and saved to `results/benchmark.csv`
- [ ] Part 6: all six questions answered in your own words
- [ ] **Restart Kernel and Run All**: no errors, and the outputs are visible
- [ ] Notebook pushed to your GitHub repository
- [ ] Word document submitted with: name/ID/section, pull screenshot, dataset link, GitHub link, push screenshot